# Session 03 — Descriptive statistics

Revision ID: S03-r1

**Question:** Which summary best describes a typical incident resolution time when values are unevenly distributed?

## Learning outcomes

1. Calculate and interpret mean, median, quartiles, range, and IQR.
2. Explain how an extreme value affects mean and standard deviation compared with median and IQR.
3. Choose suitable summaries with units and denominators.
4. Communicate a descriptive profile with limitations.

## Workflow and data

Run from the top. The exact CC0 synthetic CSV is embedded for offline use. The observational unit is one fictional closed incident. `resolution_hours` is opening-to-closure time in hours, not response latency. The export has 25 rows, one exact repeat, one negative duration, two missing durations, and one missing service. Follow the documented Session 2 policy: inspect and remove the exact repeat, change the impossible negative time to missing while retaining the incident, label the missing service `unknown`, and do not impute durations.

The export is not a random sample or a causal study. Allow time to write interpretations; running code alone is not completion.

In [1]:
import pandas as pd
from io import StringIO

csv_text = """incident_id,service,severity,resolution_hours,deploy_version,customer_impact
INC-001,api,low,2,v1,no
INC-002,web,medium,5,v1,yes
INC-003,worker,high,8,v1,no
INC-004,api,low,3,v1,yes
INC-005,web,medium,,v1,no
INC-006,worker,high,12,v1,yes
INC-007,api,low,4,v1,no
INC-008,web,medium,-2,v1,yes
INC-009,worker,high,9,v1,no
INC-010,api,low,2,v1,yes
INC-011,,medium,8,v1,no
INC-012,worker,high,15,v1,yes
INC-013,api,low,3,v2,no
INC-014,web,medium,6,v2,yes
INC-015,worker,high,10,v2,no
INC-016,api,low,4,v2,yes
INC-017,web,medium,,v2,no
INC-018,worker,high,18,v2,yes
INC-019,api,low,5,v2,no
INC-020,web,medium,10,v2,yes
INC-021,worker,high,14,v2,no
INC-022,api,low,6,v2,yes
INC-023,web,medium,11,v2,no
INC-024,worker,high,24,v2,yes
INC-003,worker,high,8,v1,no
"""
raw = pd.read_csv(StringIO(csv_text))
raw_snapshot = raw.copy(deep=True)
clean = raw.drop_duplicates().copy()
invalid = clean["resolution_hours"] < 0
clean.loc[invalid, "resolution_hours"] = float("nan")
clean["service"] = clean["service"].fillna("unknown")
assert raw.equals(raw_snapshot)
assert len(clean) == 24 and clean["resolution_hours"].count() == 21
print(f"Raw rows: {len(raw)}; distinct incidents: {len(clean)}; observed durations: {clean['resolution_hours'].count()}")

Raw rows: 25; distinct incidents: 24; observed durations: 21


## S03-E1

Compare `[2, 3, 4, 5, 6]` and `[2, 3, 4, 5, 24]`. Calculate each mean and median. Explain which summary changes more and why the value 24 should not automatically be deleted.

In [2]:
comparison = None
# TODO: calculate the mean and median for both lists and store them in a table.

## S03-E2

Using the cleaned `resolution_hours` values, calculate the number observed, median, Q1, Q3, IQR, and sample standard deviation. Include units and explain why standard deviation and IQR need not tell the same story.

In [3]:
duration_profile = None
# TODO: calculate n, median, Q1, Q3, IQR, and sample SD from nonmissing hours.

## S03-E3

Create a service-level table with incident-record count, observed-duration count, mean hours, and median hours. Write a cautious comparison that includes denominators and one limitation.

In [4]:
service_profile = None
# TODO: group by service and calculate size, count, mean, and median for resolution_hours.

## Interpret and communicate

Write one or two sentences for each exercise. State hours and the observed-value denominator. A large value is not automatically an error, and service summaries are descriptive rather than causal.

## Synthesis

The mean uses magnitudes; the median uses rank. IQR and standard deviation describe spread in different ways. Standard deviation is not a standard error. Keep group counts beside summaries so readers can see how many observed values support them.

## Readings

- [Learning Statistics with Python, Chapter 5](https://ethanweed.github.io/pythonbook/05.01-descriptive_statistics.html)
- [Think Stats, Chapter 1](https://allendowney.github.io/ThinkStats/chap01.html)


## Frequency tables and measurement scales

A count is appropriate for categorical variables. `service` is nominal; `severity` is ordered; `incident_id` is an identifier, not a measurement. For a categorical frequency table, show both counts and proportions, and state the denominator. `dropna=False` preserves an unlabelled category when one is present.

## Frequency tables and measurement scales

A count is appropriate for categorical variables. `service` is nominal; `severity` is ordered; `incident_id` is an identifier, not a measurement. For a categorical frequency table, show both counts and proportions, and state the denominator. `dropna=False` preserves an unlabelled category when one is present.

## S03 worked example: categorical frequencies

Use counts to summarize the nominal `service` label. A proportion uses a denominator: here the 24 distinct incident records, including the `unknown` service category. `severity` is ordered, but the codes are labels rather than equally spaced numbers.

In [5]:
service_records = clean["service"].value_counts(dropna=False).rename("records")
service_proportions = clean["service"].value_counts(normalize=True, dropna=False).rename("proportion")
service_frequencies = pd.concat([service_records, service_proportions], axis=1)
display(service_frequencies)
severity_frequencies = clean["severity"].value_counts().reindex(["low", "medium", "high"], fill_value=0)
display(severity_frequencies.to_frame("records"))

,records,proportion
service,,
api,8,0.333333
worker,8,0.333333
web,7,0.291667
unknown,1,0.041667


,records
severity,
low,8
medium,8
high,8


## Numeric profile and quantile positions

`describe()` gives a compact numerical profile. Quantiles mark positions in the ordered observed values; the 25th and 75th percentiles define Q1 and Q3. Pandas uses linear interpolation by default, so another package or quantile convention may differ slightly.

In [6]:
hours = clean["resolution_hours"].dropna()
display(hours.describe().round(2))
quantiles = hours.quantile([0.05, 0.25, 0.50, 0.75, 0.95]).rename("hours")
display(quantiles)

count    21.00
mean      8.52
std       5.70
min       2.00
25%       4.00
50%       8.00
75%      11.00
max      24.00
Name: resolution_hours, dtype: float64

0.05     2.0
0.25     4.0
0.50     8.0
0.75    11.0
0.95    18.0
Name: hours, dtype: float64

## Variance and standard deviation

Variance is the mean squared deviation from the mean; its units are squared. Standard deviation returns to the original units. Pandas uses `ddof=1` for sample variance and SD; `ddof=0` describes the finite set using divisor n. Neither quantity is a standard error.

In [7]:
dispersion = pd.Series({
    "variance, divisor n (hours squared)": hours.var(ddof=0),
    "sample variance, divisor n-1 (hours squared)": hours.var(ddof=1),
    "sample standard deviation (hours)": hours.std(ddof=1),
})
display(dispersion.round(2))

variance, divisor n (hours squared)             30.92
sample variance, divisor n-1 (hours squared)    32.46
sample standard deviation (hours)                5.70
dtype: float64

## Optional bridge: Matplotlib's bundled stock data

Matplotlib includes a local sample of Google's daily stock prices. The 1,047 trading-day records contain closing prices and share volumes from August 2004 through October 2008. This bridge is optional; the main session analysis still uses the incident data. No network access or extra dataset package is needed.

In [8]:
import numpy as np
from matplotlib.cbook import get_sample_data

with np.load(get_sample_data("goog.npz", asfileobj=False)) as sample:
    stock_data = sample["price_data"]
google = pd.DataFrame({
    "date": stock_data["date"],
    "close_usd": stock_data["close"],
    "volume_shares": stock_data["volume"],
})
print("Rows:", len(google), "Date range:", google["date"].min(), "to", google["date"].max())
display(google["close_usd"].describe().round(2))

Rows: 1047 Date range: 2004-08-19 00:00:00 to 2008-10-14 00:00:00


count    1047.00
mean      404.30
std       142.65
min       100.01
25%       299.87
50%       422.86
75%       497.42
max       741.79
Name: close_usd, dtype: float64